#### GF_Poisson_barb

In [ ]:
%run common/gaussian_denoising.ipynb

In [ ]:
from pathlib import Path
from collections import namedtuple

In [ ]:
Args = namedtuple("args", ["input", "output"])
args = Args("http://www.hpca.ual.es/~vruiz/images/barb.png",
            "GF_Poisson_barb.pdf")

In [ ]:
from my_google_auth import DriveHandler
service = DriveHandler.get_drive_service()
handler = DriveHandler.DriveHandler(service)

In [ ]:
MY_SHARED_DRIVE_ID = '1hGHvkP46fxLCQbUlyYhAS_eVl6PollQM'  # "tmp" folder

In [ ]:
for attribute_name in dir(args):
    if attribute_name.startswith('output'):
        output = getattr(args, attribute_name)
        file_id = handler.find_file_id(drive_file_name=output, drive_folder_id=MY_SHARED_DRIVE_ID)
        if file_id == None:
            print(f"{output} does not exist in Google Drive. Creating ...")
        else:
            print(f"Downloading {output} from Google Drive")
            success = handler.download(file_id, local_save_path=output)

In [ ]:
image = skimage_io.imread(args.input)  # Ground Truth
X = image

In [ ]:
gamma = 0.15
Y = np.random.poisson(X * gamma) / gamma

In [ ]:
denoiser = denoising.Monochrome_Denoising(logger)

In [ ]:
def get_gaussian_kernel(sigma=1):
    number_of_coeffs = 3
    number_of_zeros = 0
    while number_of_zeros < 2 :
        delta = np.zeros(number_of_coeffs)
        delta[delta.size//2] = 1
        coeffs = scipy.ndimage.gaussian_filter1d(delta, sigma=sigma)
        number_of_zeros = coeffs.size - np.count_nonzero(coeffs)
        number_of_coeffs += 1
    return coeffs[1:-1]

In [ ]:
sigma_kernel = 4.3
sigma = np.array([sigma_kernel, sigma_kernel])
kernel = [None]*2
kernel[0] = get_gaussian_kernel(sigma[0])
kernel[1] = get_gaussian_kernel(sigma[1])
print(np.sum(kernel[0]))
plt.plot(kernel[0])
plt.show()

In [ ]:
denoised = denoiser.filter(Y, kernel)

In [ ]:
#plt.title(r"$\mathrm{GF}_{\text{" + str(sigma_kernel) + r"}}(\mathbf{N}" + rf"/{_lambda}" + r"), \mathbf{N}\sim\mathrm{Poisson}" + rf"(\lambda={_lambda}" + r"\cdot\mathrm{Barb})$")
string  = r"$"
string += r"\mathrm{GF}"
string += r"_\text{"
string += str(sigma_kernel)
string += r"}"
string += r"(\mathbf{N}"
string += r"_{\mathcal{P}(\lambda="
string += rf"{gamma}"
string += r"\cdot\mathrm{Barb})}"
string += rf"/{gamma})"
string += r"$"
plt.title(string)
plt.imshow(denoised, cmap="gray")
plt.savefig(args.output, bbox_inches='tight')

In [ ]:
for attribute_name in dir(args):
    if attribute_name.startswith('output'):
        output = getattr(args, attribute_name)
        uploaded_file_id = handler.upload(
            local_file_path=output,
            drive_file_name=output,
            drive_folder_id=MY_SHARED_DRIVE_ID)